In [1]:
# imports
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import minimum_filter
import importlib
import data_loader
importlib.reload(data_loader)
import analyze_boundary
importlib.reload(analyze_boundary)


<module 'analyze_boundary' from '/Users/adeleyounis/Desktop/polytechnique/1A/CSC_51073/CurbDetect/src/analyze_boundary.py'>

In [2]:
# edit root paths
ROOT_PATH = project_root = Path.cwd().parent
KITTI_360_ROOT = Path(ROOT_PATH / "KITTI-360")

SEQ = "2013_05_28_drive_0000_sync"
CAM_ID = 0 # 0 = left pov, 1 right pov
SPLIT = "train"
FRAME = 250 # 325 lots of ramps, 250 lots of curb
ROAD_ID = 7
SIDEWALK_ID = 8

# experiment params
NUM_DISTANCES = 15
CANDIDATES_PER_DISTANCE = 5

# Read calibration data
proj_matrix, W, H = data_loader.load_perspective_calibration(KITTI_360_ROOT, CAM_ID)
poses = data_loader.load_cam0_to_world(KITTI_360_ROOT, SEQ)
print(f"image size: {W, H}")

img = np.array(Image.open(data_loader.get_img_path(KITTI_360_ROOT, SEQ, CAM_ID, FRAME)))

image size: (1408, 376)


In [3]:
# chunk and process point clouds
chunks = data_loader.list_static_chunks(KITTI_360_ROOT, SPLIT, SEQ)
print(len(chunks), "static chunks")

cloud = data_loader.load_cloud_for_frame(chunks, FRAME)

def visible_mask(u, v, z, W, H, k=5, tol=0.7): 
    depth = np.full((H, W), np.inf)
    u_i, v_i = u.astype(int), v.astype(int)
    order = np.argsort(-z) # far to near pts, nearest written last
    depth[v_i[order], u_i[order]] = z[order]
    local_min = minimum_filter(depth, size=k)
    return z <= local_min[v_i, u_i] + tol

cam_2_world = poses[FRAME]
u, v, z, idx = data_loader.project(cloud["xyz"], cam_2_world, proj_matrix, W, H)
vis = visible_mask(u, v, z, W, H)
u, v, z, idx = u[vis], v[vis], z[vis], idx[vis]
print(f"{len(idx):,} visible points projected")


44 static chunks
0000000002_0000000385.ply: 3,201,318 points, fields: ('x', 'y', 'z', 'red', 'green', 'blue', 'semantic', 'instance', 'visible', 'confidence')
202,813 visible points projected


In [ ]:
# visualize
def label_colours(sem):
    colour_mask = np.full((len(sem), 3), 0.6) # everything_grey
    colour_mask[sem == ROAD_ID] = (0.2, 0.2, 1.0) # road = blue
    colour_mask[sem == SIDEWALK_ID] = (1.0, 0.2, 0.2) # sidewalk = red
    return colour_mask

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax in axes:
    ax.imshow(img)

ground = np.isin(cloud["semantic"][idx], [ROAD_ID, SIDEWALK_ID])
axes[1].scatter(u, v, s=1, c=label_colours(cloud["semantic"][idx]))
axes[1].set_title("projected points, colored by label (road=blue, sidewalk=red)")

depth = cloud["xyz"][idx, 2]
lo, hi = np.percentile(depth[ground], [2, 98]) if ground.any() else (depth.min(), depth.max())
sc = axes[2].scatter(u[ground], v[ground], s=1, c=depth[ground], cmap="turbo", vmin=lo, vmax=hi)
axes[2].set_title("road and sidewalk points, colored by world height z")
plt.tight_layout()
plt.show()

In [5]:
def plot_analysis(res, title="", cam_to_world=None, proj_matrix=None):    
    fig, ax = plt.subplots(1, 1, figsize=(13, 5))

    c = np.where(res["lab"][:, None] == 7, [[0.2, 0.2, 1.0]], [[1.0, 0.2, 0.2]])

    ax.scatter(res["pts"] * 100, res["z"] * 100, s=3, c=c, alpha=0.6)
    ax.plot(res["bins"][0] * 100, res["bins"][1] * 100, "k-", lw=2.0, label="median")
    ax.axvline(res["s_step"] * 100, color="tab:red", ls="--", label="transition midpoint")
    ax.plot(res["fit_curve"][0] * 100, res["fit_curve"][1] * 100, color="darkorange", lw=2.5, label="Point Cloud fit")
    half_width = res["ramp_width_cm"] / 2
    ax.axvspan(res["s_step"] * 100 - half_width, res["s_step"] * 100 + half_width, color="orange", alpha=0.12, label="ramp region")
    ax.set_xlabel("distance across boundary (cm)")
    ax.set_ylabel("world z (cm)")
    ax.set_title(f"{title} height={res['step_cm']:.1f} cm")

    if cam_to_world is not None and proj_matrix is not None:
        side = analyze_boundary.get_curb_side(res, cam_to_world, proj_matrix)

        if side == "left":
            ax.invert_xaxis()

        if side is not None:
            ax.set_title(f"{side.title()} curb | {ax.get_title()}")
    
    ax.legend()


In [6]:
def run_experiment(n, candidate_num, plot=True):
    # u, v, z, idx were calculated for frame in the projection cell
    ground_visible = np.isin(cloud["semantic"][idx], [ROAD_ID, SIDEWALK_ID])
    ground_depths = z[ground_visible]

    target_depths = np.linspace(*np.percentile(ground_depths, [10, 90]), n)

    results_by_distance = []

    if plot:
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(img)
        colors = plt.cm.turbo(np.linspace(0.1, 0.9, n))

    for i, target in enumerate(target_depths, start=1):
        band = ground_visible & (np.abs(z - target) <= 2.0)
        original_indices = idx[band]
        depth_cloud = {k: val[original_indices] for k, val in cloud.items()}

        anchors = analyze_boundary.boundary_candidates(
            depth_cloud, ROAD_ID, SIDEWALK_ID, n=candidate_num
        )

        result = None
        selected_anchor = None

        for anchor in anchors:
            candidate = analyze_boundary.boundary_estimator(
                depth_cloud, anchor, ROAD_ID, SIDEWALK_ID)
            if candidate is not None:
                result = candidate
                selected_anchor = anchor
                break

        # store all results and fitted parameters
        results_by_distance.append({
            "depth": target,
            "height_cm": result["step_cm"] if result else np.nan,
            "sidewalk_slope": result["sidewalk_slope"] if result else np.nan,
            "fit": result.get("fit") if result else None,
            "result": result,
        })

        height = result["step_cm"] if result else np.nan
        print(f"#{i:02d}  depth={target:5.1f} m  height={height:6.1f} cm")

        if plot and selected_anchor is not None:
            nearest = np.argmin(
                np.sum((depth_cloud["xyz"][:, :2] - selected_anchor)**2, axis=1)
            )
            anchor_uv = np.flatnonzero(band)[nearest]
            color = colors[i - 1]

            ax.scatter(
                u[anchor_uv], v[anchor_uv], s=55,
                color=color, edgecolor="black", linewidth=0.7
            )
            ax.text(
                u[anchor_uv] + 6, v[anchor_uv], str(i),
                color="white", fontsize=10, weight="bold",
                bbox=dict(facecolor="black", alpha=0.65, pad=1)
            )

    if plot:
        ax.set_title(f"Frame {FRAME}: curb analysis at up to {n} camera distances")
        ax.set_xlim(0, W)
        ax.set_ylim(H, 0)
        plt.tight_layout()
        plt.show()

        for item in results_by_distance:
            if item["result"] is not None:
                plot_analysis(
                    item["result"],
                    title=f"Depth {item['depth']:.1f} m",
                    cam_to_world=cam_2_world,
                    proj_matrix=proj_matrix
                )
        plt.show()

    return results_by_distance

In [ ]:
results = run_experiment(n=10, candidate_num=5, plot=True)